####**Base de Datos (Database)**

####Objetivos
1. Documentacion sobre Spark SQL
2. Crear la Base de Datos "**demo**"
3. Acceder al "**Catalog**" en la "**Interfaz del Usuario**"
4. Comando "**SHOW**"
5. Comando **DESCRIBE (DESC)**
6. Mostrar la Base de Datos actual

In [0]:
CREATE SCHEMA IF NOT EXISTS demo;

In [0]:
SHOW DATABASES;

In [0]:
DESCRIBE DATABASE demo;

In [0]:
DESCRIBE DATABASE EXTENDED demo;

In [0]:
SELECT current_database();

In [0]:
SHOW TABLES;

In [0]:
SHOW TABLES IN demo;

In [0]:
USE demo;

In [0]:
SELECT current_database();

In [0]:
SHOW TABLES IN default;

####**Tablas Administradas (Managed Tables)**

####Objetivos
1. Crear una "**Tabla Administrada (Managed Table)**" con Python
2. Crear una "**Tabla Administrada (Managed Table)**" con SQL
3. Efecto de eliminar una Tabla Administrada
4. Describir (Describe) la tabla

In [0]:
%run "../Includes/configuration"

In [0]:
%python
results_movie_genre_language = spark.read.parquet(f"{gold_folder_path}/results_movie_genre_language")


In [0]:
%python
display(results_movie_genre_language)

In [0]:
%python
results_movie_genre_language.write.format("delta").saveAsTable("demo.results_movie_genre_language_python")

In [0]:
USE demo;
SHOW TABLES;

In [0]:
DESCRIBE results_movie_genre_language_python;

In [0]:
DESCRIBE EXTENDED results_movie_genre_language_python;

In [0]:
SELECT *
FROM results_movie_genre_language_python
WHERE genre_Name = "Adventure"

In [0]:
CREATE TABLE demo.results_movie_genre_language_sql
AS
SELECT *
FROM results_movie_genre_language_python
WHERE genre_Name = "Adventure";


In [0]:
SELECT *
FROM results_movie_genre_language_sql;

In [0]:
SELECT current_database();

In [0]:
DESCRIBE EXTENDED results_movie_genre_language_sql;

In [0]:
DROP TABLE IF EXISTS results_movie_genre_language_sql;

In [0]:
SHOW TABLES IN demo;

####**Tablas Externas (External Tables)**

####Objetivos
1. Crear una "**Tabla Externa (External Table)**" con Python
2. Crear una "**Tabla Externa (External Table)**" con SQL
3. Efecto de eliminacion de una "**Tabla Externa (External Table)**"
4. Describir (Describe) la tabla

In [0]:
%python
results_movie_genre_language.write.format("parquet").option("path", f"{gold_folder_path}/results_movie_genre_language_py").saveAsTable("demo.results_movie_genre_language_py")

In [0]:
DESCRIBE EXTENDED demo.results_movie_genre_language_py;

In [0]:
SELECT * FROM demo.results_movie_genre_language_py;

In [0]:
CREATE TABLE demo.results_movie_genre_language_sql(
    title STRING,
    duration_Time INT,
    release_Date DATE,
    vote_Average FLOAT,
    Language_Name STRING,
    genre_Name STRING,
    created_date TIMESTAMP
)
USING PARQUET
LOCATION "abfss://gold@moviehistory310785.dfs.core.windows.net/results_movie_genre_language_ext_sql"

In [0]:
/*La tabla creada se verifica que ya figura*/

SHOW TABLES IN demo;

In [0]:
/*La carpeta aun se visualiza en el contenedor gold debido a que la tabla esta vacia, cuando se llene de registros se hara un update en el contenedor y recien se visualizara*/

SELECT * FROM demo.results_movie_genre_language_sql

In [0]:
/* Luego se hace update en el contenedor y ya deberia mostrarse la carpeta
"abfss://gold@moviehistory31071985.dfs.core.windows.net/results_movie_genre_language_ext_sql" */

INSERT INTO demo.results_movie_genre_language_sql
SELECT * FROM demo.results_movie_genre_language_py
WHERE genre_Name = "Adventure";

In [0]:
SELECT COUNT(1) FROM demo.results_movie_genre_language_sql

In [0]:
SHOW TABLES IN demo;

In [0]:
DROP TABLE demo.results_movie_genre_language_sql;

In [0]:
/*La tabla se elimino pero porser almacenadas en una ubicacion externa los datos se mantienen, se puede validar en la carpeta contenedora que aun figuran*/

SHOW TABLES IN demo;

####Vistas **(Views)**

####Objetivos
1. Crear Vista Temporal
2. Crear Vista Temporal Global
3. Crear Vista Permamente

In [0]:
SELECT current_database();

In [0]:
CREATE OR REPLACE TEMP VIEW v_results_movies_genres_language 
AS 
SELECT * 
FROM demo.results_movie_genre_language_py 
WHERE genre_Name = "Adventure"

In [0]:
SELECT * FROM v_results_movies_genres_language;

In [0]:
CREATE OR REPLACE GLOBAL TEMP VIEW gv_results_movies_genres_language
AS 
SELECT * 
FROM demo.results_movie_genre_language_py 
WHERE genre_Name = "Drama";

In [0]:
SHOW TABLES;

In [0]:
SHOW TABLES IN global_temp;

In [0]:
SELECT * FROM global_temp.gv_results_movies_genres_language

In [0]:
CREATE OR REPLACE VIEW pv_results_movies_genres_language
AS 
SELECT * 
FROM demo.results_movie_genre_language_py 
WHERE genre_Name = "Comedy";

In [0]:
SHOW TABLES;

In [0]:
SELECT * FROM pv_results_movies_genres_language;